# Step 1 — Entity BIO labeling (FOOD_ITEM/DRINK_ITEM/QUANTITY/MODIFIER taxonomy)

`data/normalized/augmented_ordering_dataset_normalized.jsonl` already carries a gold `entities` field per record — each entry is `{"label": <FOOD_ITEM|DRINK_ITEM|QUANTITY|MODIFIER>, "token": <span text>, "start": <char offset>, "end": <char offset>}`. The `start`/`end` offsets are computed against the raw `text` field (pre-normalization casing/punctuation), not `text_normalized`, so they can't be used directly — this notebook converts the gold spans into token-level BIO tags over `text_normalized` via word-token matching instead (mirrors `3.1_disfluency_bio_labeling.ipynb`'s `match_spans` approach).

In [1]:
import json
from collections import Counter
from pathlib import Path

SRC = Path("../data/normalized/augmented_ordering_dataset_normalized.jsonl")
OUT_DIR = Path("../data/ner")

## Step 2 — Span matching helpers

Gold `token` text doesn't tokenize cleanly against `text_normalized.split()` for two reasons: (1) hyphenated dish names (`Gado-Gado`) split into separate words after normalization (`gado gado`), and (2) Indonesian clitics glue onto the adjacent word during normalization (`bandrek` -> `bandreknya`, `tambah` -> `ditambah`, `level tiga` -> written-out in gold but the text keeps it as-is). `norm_word` strips punctuation, lowercases, and maps spelled-out digit words (`satu`..`sepuluh`) to digits so `level 3` (gold) matches `level tiga` (text) after normalization. Matching first tries an exact token-window match; if that fails, it retries allowing the *first and last* word of the span to be a prefix/suffix match instead of exact equality, to absorb glued clitics at the span's edges.

In [2]:
import re

NUM_WORDS = {
    "satu": "1", "dua": "2", "tiga": "3", "empat": "4", "lima": "5",
    "enam": "6", "tujuh": "7", "delapan": "8", "sembilan": "9", "sepuluh": "10",
}


def tokenize(text: str) -> list[str]:
    return text.split()


def norm_word(tok: str) -> str:
    w = re.sub(r"[^a-z0-9]", "", tok.lower())
    return NUM_WORDS.get(w, w)


def get_span_words(token: str) -> list[str]:
    parts = re.split(r"[\s-]+", token)
    return [norm_word(p) for p in parts if p]

## Step 3 — BIO labeler from gold spans

`match_spans` locates each gold entity's word span inside the sentence and marks those positions used, so repeated surface forms bind to the correct occurrence instead of always the first. Entries aren't guaranteed to be in left-to-right text order, so matching is order-independent over the entities list rather than a single sequential scan (same rationale as `3.1`).

In [3]:
def _window_matches(toks: list[str], i: int, span: list[str]) -> bool:
    """Exact match of a normalized word window against the gold span."""
    return [norm_word(t) for t in toks[i : i + len(span)]] == span


def _window_matches_fuzzy(toks: list[str], i: int, span: list[str]) -> bool:
    """Like _window_matches, but the first/last word may be a prefix or
    suffix match instead of exact equality, to absorb glued clitics
    (e.g. "bandrek" gold vs "bandreknya" in text_normalized)."""
    L = len(span)
    for j in range(L):
        w = norm_word(toks[i + j])
        g = span[j]
        if w == g:
            continue
        if (j == 0 or j == L - 1) and (w.startswith(g) or w.endswith(g)):
            continue
        return False
    return True


def match_spans(toks: list[str], entities: list[dict]) -> list[tuple[int, int, str]] | None:
    """Match each gold entity entry to a word span, returning (start, end, label)
    triples (end exclusive). Returns None if any entry can't be matched."""
    used = [False] * len(toks)
    spans = []
    for e in entities:
        span = get_span_words(e["token"])
        L = len(span)
        found = -1
        for matcher in (_window_matches, _window_matches_fuzzy):
            for i in range(len(toks) - L + 1):
                if any(used[i : i + L]):
                    continue
                if matcher(toks, i, span):
                    found = i
                    break
            if found != -1:
                break
        if found == -1:
            return None
        for k in range(found, found + L):
            used[k] = True
        spans.append((found, found + L, e["label"]))
    return spans


def full_label(raw_text: str, entities: list[dict]) -> tuple[list[str], list[str] | None]:
    toks = tokenize(raw_text)
    labels = ["O"] * len(toks)
    spans = match_spans(toks, entities)
    if spans is None:
        return toks, None
    for start, end, label in spans:
        labels[start] = f"B-{label}"
        for k in range(start + 1, end):
            labels[k] = f"I-{label}"
    return toks, labels

## Step 4 — Validation

Auto-corrects orphan `I-` tags (continuation tag with no preceding `B-`/`I-` of the same type) to `B-`, and any unrecognized label value to `O`. `match_spans`/`full_label` never produce orphans by construction, but this guards against future changes silently breaking BIO well-formedness (same convention as `3.1`).

In [4]:
ENTITY_TAGS = ["FOOD_ITEM", "DRINK_ITEM", "QUANTITY", "MODIFIER"]
VALID_LABELS = {"O"} | {f"{p}-{t}" for t in ENTITY_TAGS for p in ("B", "I")}


def validate_bio(toks: list[str], labels: list[str], record_id) -> list[str]:
    fixed = list(labels)
    prev_type = None
    for i, lb in enumerate(fixed):
        if lb not in VALID_LABELS:
            print(f"[{record_id}] unknown label {lb!r} at token {toks[i]!r}, coercing to O")
            fixed[i] = "O"
            lb = "O"
        if lb.startswith("I-"):
            tag_type = lb[2:]
            if prev_type != tag_type:
                print(f"[{record_id}] orphan {lb} at token {toks[i]!r}, coercing to B-{tag_type}")
                fixed[i] = f"B-{tag_type}"
        prev_type = fixed[i][2:] if fixed[i] != "O" else None
    return fixed

## Step 5 — Run labeler over the corpus

In [5]:
rows = [json.loads(line) for line in SRC.open()]

labeled_rows = []
tag_counts = Counter()
unmatched = []
for row in rows:
    toks, labels = full_label(row["text_normalized"], row.get("entities", []))
    if labels is None:
        unmatched.append(row["id"])
        continue
    labels = validate_bio(toks, labels, row["id"])
    for lb in labels:
        tag_counts[lb] += 1
    labeled_rows.append({
        "id": row["id"],
        "text": row["text_normalized"],
        "intent": row["intent"],
        "tokens": toks,
        "labels": labels,
    })

print(f"{len(labeled_rows)} records labeled, {len(unmatched)} unmatched: {unmatched}")
print("tag counts:", dict(tag_counts))

6021 records labeled, 11 unmatched: ['75_aug2', '177_aug2', '254_aug1', 484, '484_aug1', '484_aug2', 877, '877_aug1', '877_aug2', '1094_aug1', '925_aug2']
tag counts: {'O': 102469, 'B-FOOD_ITEM': 11093, 'B-QUANTITY': 2130, 'B-DRINK_ITEM': 2925, 'I-DRINK_ITEM': 2903, 'B-MODIFIER': 4850, 'I-MODIFIER': 5446, 'I-FOOD_ITEM': 11654}


## Step 6 — Sanity check on sample records

In [6]:
sample_ids = [1, 2, "1_aug1", 7]
samples_by_id = {row["id"]: row for row in rows}

for rid in sample_ids:
    row = samples_by_id[rid]
    toks, labels = full_label(row["text_normalized"], row.get("entities", []))
    print(row["text_normalized"])
    print([f"{t}/{l}" for t, l in zip(toks, labels) if l != "O"] or "no tags")
    print()

eh mbak toiletnya di mana ya yang anu gak ketemu ar ar
no tags

kalau boleh eh jangan deh pesenannya bukan bakmi kuah tapi indomie kuah ya tidak pakai daun bawang juga pak
['bakmi/B-FOOD_ITEM', 'kuah/I-FOOD_ITEM', 'indomie/B-FOOD_ITEM', 'kuah/I-FOOD_ITEM', 'tidak/B-MODIFIER', 'pakai/I-MODIFIER', 'daun/I-MODIFIER', 'bawang/I-MODIFIER']

sebentar sebentar maksudnya saya mau ganti porsi bakmi kuah jadi dua ya bukan satu tadi jus jeruk juga masih satu aja tambah udang semua ya
['bakmi/B-FOOD_ITEM', 'kuah/I-FOOD_ITEM', 'dua/B-QUANTITY', 'satu/B-QUANTITY', 'jus/B-DRINK_ITEM', 'jeruk/I-DRINK_ITEM', 'satu/B-QUANTITY', 'tambah/B-MODIFIER', 'udang/I-MODIFIER']

eh bisa gak diganti siomaynya jadi gudeg ya eh tambah deh nambahin susu putih juga mungkin tambah bawang goreng juga ya
['siomaynya/B-FOOD_ITEM', 'gudeg/B-FOOD_ITEM', 'susu/B-DRINK_ITEM', 'putih/I-DRINK_ITEM', 'tambah/B-MODIFIER', 'bawang/I-MODIFIER', 'goreng/I-MODIFIER']



## Step 7 — Align labels to IndoBERT WordPiece tokens

`indobenchmark/indobert-base-p1` — matches the disfluency-tagging backbone (`3.1`/`3.2`). Word-level BIO labels align to WordPiece subwords via `word_ids()`: the first subword of each word keeps its label id, continuation subwords and special tokens get `-100` so `CrossEntropyLoss`/CRF training ignores them.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = ["O"] + [f"{p}-{t}" for t in ENTITY_TAGS for p in ("B", "I")]
label2id = {lb: i for i, lb in enumerate(label_list)}
id2label = {i: lb for lb, i in label2id.items()}
print(label2id)

{'O': 0, 'B-FOOD_ITEM': 1, 'I-FOOD_ITEM': 2, 'B-DRINK_ITEM': 3, 'I-DRINK_ITEM': 4, 'B-QUANTITY': 5, 'I-QUANTITY': 6, 'B-MODIFIER': 7, 'I-MODIFIER': 8}


In [8]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    word_ids = encoding.word_ids()
    aligned = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != prev_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(-100)
        prev_word_id = word_id
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": aligned,
    }


aligned_rows = []
mismatches = 0
for r in labeled_rows:
    aligned = align_labels_with_tokens(r["tokens"], r["labels"])
    if len(aligned["input_ids"]) != len(aligned["labels"]):
        mismatches += 1
        continue
    aligned_rows.append({
        "id": r["id"],
        "intent": r["intent"],
        **aligned,
    })

print(f"{len(aligned_rows)} aligned, {mismatches} mismatches")

6021 aligned, 0 mismatches


## Step 8 — Dominant entity type per record (for stratified split)

Each record gets one dominant tag for stratification — the rarest non-`O` tag type present, falling back to `none` for records with no entity tags. Prioritizing the rarest tag keeps `MODIFIER` (the least common type) from being randomly orphaned into a single split.

In [9]:
TAG_PRIORITY = ["MODIFIER", "QUANTITY", "DRINK_ITEM", "FOOD_ITEM"]


def dominant_type(labels: list[str]) -> str:
    present = {lb[2:] for lb in labels if lb != "O"}
    for tag in TAG_PRIORITY:
        if tag in present:
            return tag
    return "none"


dominant_by_id = {r["id"]: dominant_type(r["labels"]) for r in labeled_rows}
print(Counter(dominant_by_id.values()))

Counter({'MODIFIER': 2430, 'FOOD_ITEM': 385, 'DRINK_ITEM': 304, 'QUANTITY': 138, 'none': 57})


## Step 9 — Stratified 80/10/10 split

This runs as *two* sequential splits (80/20, then the 20% into 10/10) — a class needs >=4 total occurrences to survive both with `train_test_split`'s >=2-members-per-split requirement. Records whose dominant type has fewer than 4 occurrences go entirely to train; the rest stratify normally (same convention as `3.1`).

In [10]:
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
MIN_STRATIFY_COUNT = 4

counts = Counter(dominant_by_id.values())
forced_train_ids = {rid for rid, dt in dominant_by_id.items() if counts[dt] < MIN_STRATIFY_COUNT}
stratifiable = [r for r in aligned_rows if r["id"] not in forced_train_ids]
forced_train = [r for r in aligned_rows if r["id"] in forced_train_ids]

labels_strat = [dominant_by_id[r["id"]] for r in stratifiable]
train, rest = train_test_split(
    stratifiable, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_labels = [dominant_by_id[r["id"]] for r in rest]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_labels, random_state=SEED
)
train = train + forced_train

print(f"train={len(train)} val={len(val)} test={len(test)}")

train=4214 val=903 test=904


In [11]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/ner
